## 1. Imports and directory setup

In this section, we import the required libraries for building deep learning models (TensorFlow/Keras) and define the folder paths. We also set random seeds to ensure the results are reproducible.

- `../data/processed` contains the cleaned data and feature metadata from the preprocessing step.
- `../models` will be used to save the trained model checkpoints.

In [ ]:
# Imports + Folders
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from scipy.fft import fft
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

(574780, 341) (107, 336)


## 2. Load processed datasets and metadata

We load the `train_clean.pkl` dataset and the `feature_metadata.pkl` dictionary created in the previous notebook. The metadata is critical because it tells us which columns belong to the Time-of-Flight (TOF) sensors and which belong to the IMU/Thermopile sensors, allowing us to split the inputs for the different models.

In [ ]:
# Load cleaned data and metadata
with open(DATA_PROCESSED / "train_clean.pkl", "rb") as f:
    train_df = pickle.load(f)

with open(DATA_PROCESSED / "feature_metadata.pkl", "rb") as f:
    meta = pickle.load(f)

# Extract column groups
tof_cols = meta["tof_cols"]
imu_thm_cols = meta["imu_thm_cols"]

print("Train shape:", train_df.shape)
print("TOF columns:", len(tof_cols))
print("IMU/Thermopile columns:", len(imu_thm_cols))

# Encode string labels to integers
label_encoder = LabelEncoder()
unique_gestures = train_df['gesture'].unique()
label_encoder.fit(unique_gestures)
num_classes = len(unique_gestures)

print(f"Unique classes: {num_classes}")

## 3. Data Generation and Transformation

We implement a custom `Keras Sequence` generator to handle the complex data transformations required by the paper.

The generator performs the following steps on-the-fly during training:
1.  **Grouping:** Groups individual time-steps by `sequence_id`.
2.  **FFT Transformation:** Applies Fast Fourier Transform to IMU/Thermopile data for the MLP model.
3.  **Reshaping:** Reshapes flat TOF data into 5-channel $8 \times 8$ images for the CNN model.
4.  **Padding:** Standardizes sequence lengths.

In [ ]:
class BFRBDataGenerator(keras.utils.Sequence):
    def __init__(self, df, sequence_ids, batch_size=256, max_len=100, shuffle=True, n_classes=18):
        self.df = df
        self.sequence_ids = sequence_ids
        self.batch_size = batch_size
        self.max_len = max_len
        self.shuffle = shuffle
        self.n_classes = n_classes
        self.indexes = np.arange(len(self.sequence_ids))
        
        # Group data once for faster access during training
        print("Grouping data by sequence... (this may take a few seconds)")
        self.grouped = dict(tuple(df.groupby('sequence_id')))
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.sequence_ids) / self.batch_size))

    def __getitem__(self, index):
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        batch_seq_ids = [self.sequence_ids[k] for k in batch_indexes]
        return self.__data_generation(batch_seq_ids)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, batch_seq_ids):
        X_fft = []
        X_tof = []
        y = []

        for seq_id in batch_seq_ids:
            seq_data = self.grouped[seq_id]
            
            # Label
            label_str = seq_data['gesture'].iloc[0]
            y.append(label_encoder.transform([label_str])[0])

            # 1. FFT Features (IMU + Thermopile)
            ts_data = seq_data[imu_thm_cols].values
            # Pad or truncate to max_len
            if len(ts_data) < self.max_len:
                pad = ((0, self.max_len - len(ts_data)), (0, 0))
                ts_padded = np.pad(ts_data, pad, mode='constant')
            else:
                ts_padded = ts_data[:self.max_len]
            # Apply FFT and take magnitude
            fft_features = np.abs(fft(ts_padded, axis=0)).flatten()
            X_fft.append(fft_features)

            # 2. TOF Images
            tof_flat = seq_data[tof_cols].values
            if len(tof_flat) < self.max_len:
                pad = ((0, self.max_len - len(tof_flat)), (0, 0))
                tof_padded = np.pad(tof_flat, pad, mode='constant')
            else:
                tof_padded = tof_flat[:self.max_len]
            
            # Reshape to (Time, 5 sensors, 8 rows, 8 cols) -> Then to channels last (Time, 8, 8, 5)
            tof_reshaped = tof_padded.reshape(self.max_len, 5, 8, 8).transpose(0, 2, 3, 1)
            X_tof.append(tof_reshaped)

        # CHANGE HERE: Use a Tuple () instead of a List [] for the inputs
        return (np.array(X_fft), np.array(X_tof)), keras.utils.to_categorical(y, num_classes=self.n_classes)

# Prepare Train/Val Split
all_seqs = train_df['sequence_id'].unique()
train_ids, val_ids = train_test_split(all_seqs, test_size=0.2, random_state=RANDOM_SEED)

print("Training sequences:", len(train_ids))
print("Validation sequences:", len(val_ids))

# Initialize Generators
gen_train = BFRBDataGenerator(train_df, train_ids, batch_size=256, max_len=100, n_classes=num_classes)
gen_val = BFRBDataGenerator(train_df, val_ids, batch_size=256, max_len=100, shuffle=False, n_classes=num_classes)

# Check input dimension from a sample batch
X_sample, y_sample = gen_train[0]
fft_dim = X_sample[0].shape[1]
print("FFT Input Dimension:", fft_dim)

## 4. Define and Train Ensemble Model

We define the architecture specified in the paper:
1.  **FFT-MLP:** A Multilayer Perceptron for the frequency features.
2.  **CNN-BiLSTM:** A convolutional recurrent network for the TOF image sequences.
3.  **Ensemble:** A Late Fusion layer that concatenates the features before the final softmax classification.

In [ ]:
def build_ensemble(fft_dim, max_len, num_classes):
    # --- Branch 1: FFT-MLP ---
    fft_input = keras.Input(shape=(fft_dim,), name="fft_input")
    x1 = layers.Dense(128, activation="relu")(fft_input)
    x1 = layers.Dense(64, activation="relu")(x1)
    mlp_features = layers.Dense(64, activation="relu")(x1) # Feature layer

    # --- Branch 2: CNN-BiLSTM ---
    tof_input = keras.Input(shape=(max_len, 8, 8, 5), name="tof_input")
    # TimeDistributed CNN
    x2 = layers.TimeDistributed(layers.Conv2D(32, (3,3), padding='same', activation='relu'))(tof_input)
    x2 = layers.TimeDistributed(layers.Conv2D(64, (3,3), padding='same', activation='relu'))(x2)
    x2 = layers.TimeDistributed(layers.Flatten())(x2)
    x2 = layers.TimeDistributed(layers.Dense(128, activation='relu'))(x2)
    # BiLSTM
    bilstm_features = layers.Bidirectional(layers.LSTM(128))(x2)

    # --- Late Fusion ---
    combined = layers.Concatenate()([mlp_features, bilstm_features])
    output = layers.Dense(num_classes, activation="softmax", name="output")(combined)

    model = keras.Model(inputs=[fft_input, tof_input], outputs=output, name="Ensemble_Model")
    
    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

# Build and Train
model = build_ensemble(fft_dim, 100, num_classes)
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint(str(MODELS_DIR / "ensemble_best.keras"), save_best_only=True, monitor="val_accuracy"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
]

history = model.fit(
    gen_train,
    validation_data=gen_val,
    epochs=20,
    callbacks=callbacks
)

# Plot Results
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Val')
plt.title('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.title('Loss')
plt.legend()
plt.show()